In [ ]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import spearmanr  

# Load Data
df = pd.read_csv('feature vector with class')

# Drop unnecessary columns
df = df.drop(['Unnamed: 0', 'class'], axis=1)

# Ensure 'SUBID' is retained
if 'SUBID' not in df.columns:
    raise ValueError("SUBID column is missing from the dataframe.")

# Dictionary to store correlation matrices for each SUBID
all_correlations = {}

# Get list of brain region columns (excluding SUBID)
brain_regions = [col for col in df.columns if col != 'SUBID']

# Iterate through each subject
for index, row in df.iterrows():
    subid = row['SUBID']  
    
    # Convert comma-separated strings to lists of floats (if necessary)
    values = {}
    for region in brain_regions:
        if isinstance(row[region], str):  
            values[region] = list(map(float, row[region].split(',')))
        else:
            values[region] = row[region]  
    
    # Create correlation matrix
    n_regions = len(brain_regions)
    correlation_matrix = np.zeros((n_regions, n_regions))
    
    for i in range(n_regions):
        for j in range(n_regions):
            if len(set(values[brain_regions[i]])) == 1 or len(set(values[brain_regions[j]])) == 1:
                correlation_matrix[i, j] = 0  
            else:
                corr, _ = spearmanr(values[brain_regions[i]], values[brain_regions[j]])  
                correlation_matrix[i, j] = corr
    
    # Store correlation matrix as a DataFrame
    correlation_df = pd.DataFrame(correlation_matrix, index=brain_regions, columns=brain_regions)
    all_correlations[subid] = correlation_df
    
    # Save each subject's correlation matrix
    correlation_df.to_excel(f'correlation_matrix_spearman_subid_{subid}.xlsx')

print("Spearman correlation matrices saved for all subjects.")

# Function to create thresholded brain network
def create_brain_network(correlation_matrix, subid, threshold=0):
    G_threshold = nx.Graph()
    brain_regions = correlation_matrix.columns
    G_threshold.add_nodes_from(brain_regions)

    total_possible_edges = len(brain_regions) * (len(brain_regions) - 1) / 2
    remaining_edges = 0

    # Thresholding: Remove weak edges
    for i in range(len(brain_regions)):
        for j in range(i+1, len(brain_regions)):
            weight = correlation_matrix.iloc[i, j]
            if abs(weight) >= threshold:
                G_threshold.add_edge(brain_regions[i], brain_regions[j], weight=weight)
                remaining_edges += 1
    
    # Save thresholded correlation matrix
    correlation_matrix[abs(correlation_matrix) < threshold] = 0
    correlation_matrix.to_excel(f'thresholded_correlation_matrix_spearman_subid_{subid}.xlsx')
    
    # Calculate percentage of edges removed
    percentage_removed = (1 - (remaining_edges / total_possible_edges)) * 100
    print(f'SUBID {subid}: {percentage_removed:.2f}% of edges removed.')

    # Plot graph
    plt.figure(figsize=(12, 12))
    pos = nx.spring_layout(G_threshold, k=0.5, iterations=50)
    nx.draw(G_threshold, pos, node_color='lightblue', node_size=500, font_size=8, font_weight='bold',
            edge_color='gray', width=[G_threshold[u][v]['weight'] for u, v in G_threshold.edges()], with_labels=True)
    plt.title(f'Thresholded Brain Network (Threshold={threshold}) - SUBID {subid}')
    plt.savefig(f'thresholded_network_spearman_subid_{subid}.png', dpi=300)
    plt.show()

    return G_threshold, correlation_matrix 
    
# Create thresholded networks for each subject
thresholded_graphs = {}
for subid, matrix in all_correlations.items(): 
    G, M = create_brain_network(matrix, subid, 0)
    all_correlations[subid] = M
    thresholded_graphs[subid] = G

print("Thresholded networks using Spearman created and saved.")


# Function to compute graph metrics (No Normalization Applied)
def calculate_node_metrics(G, subid):
    if len(G.nodes) == 0:  
        return []

    node_metrics = []

    # Convert all weights to absolute values (keep this)
    G_abs = G.copy()
    for u, v, d in G_abs.edges(data=True):
        d['weight'] = abs(d['weight'])  

    # Compute centrality measures (after thresholding)
    degree_centrality = nx.degree_centrality(G_abs)
    betweenness_centrality = nx.betweenness_centrality(G_abs, weight='weight')
    clustering_coefficient = nx.clustering(G_abs, weight='weight')
    closeness_centrality = nx.closeness_centrality(G_abs, distance='weight')
    strength = {node: sum(G_abs[node][nbr]['weight'] for nbr in G_abs[node]) for node in G_abs.nodes()}

    # New: Compute PageRank Centrality
    pagerank_centrality = nx.pagerank(G_abs, weight='weight')

    # New: Compute Eigenvector Centrality (May fail for disconnected graphs)
    try:
        eigenvector_centrality = nx.eigenvector_centrality_numpy(G_abs, weight='weight')
    except nx.NetworkXError:
        eigenvector_centrality = {node: 0 for node in G_abs.nodes()}  

    # New: Compute Shortest Path Lengths
    shortest_paths = dict(nx.all_pairs_dijkstra_path_length(G_abs, weight='weight'))

    # Compute Participation Coefficient
    for node in G_abs.nodes():
        k = G_abs.degree(node)
        k_c = {}
        for neighbor in G_abs[node]:
            c = neighbor  
            k_c[c] = k_c.get(c, 0) + 1

        participation_coefficient = 1 - sum((k_ci / k) ** 2 for k_ci in k_c.values()) if k > 0 else 0

        # Compute Average Shortest Path Length from this node
        avg_shortest_path_length = np.mean(list(shortest_paths[node].values())) if node in shortest_paths else float('inf')

        node_metrics.append({
            'SUBID': subid,
            'Node': node,
            'Degree_Centrality': degree_centrality[node],
            'Betweenness_Centrality': betweenness_centrality[node],
            'Clustering_Coefficient': clustering_coefficient[node],
            'Strength': strength[node],
            'Closeness_Centrality': closeness_centrality[node],
            'Participation_Coefficient': participation_coefficient,
            'PageRank_Centrality': pagerank_centrality[node],
            'Eigenvector_Centrality': eigenvector_centrality[node],
            'Avg_Shortest_Path_Length': avg_shortest_path_length
        })
    
    return node_metrics

# Compute metrics for all subjects
all_node_metrics = []
for subid, G in thresholded_graphs.items():
    all_node_metrics.extend(calculate_node_metrics(G, subid))

# Convert to DataFrame and save
metrics_df = pd.DataFrame(all_node_metrics)
metrics_df.to_csv('node_metrics_by_subid_0.5threshold.csv', index=False)

# Reshape metrics into subject-wise format
reshaped_df = metrics_df.pivot_table(index='SUBID', columns='Node',
                                     values=['Degree_Centrality', 'Betweenness_Centrality', 
                                             'Clustering_Coefficient', 'Strength', 
                                             'Closeness_Centrality', 'Participation_Coefficient',
                                             'PageRank_Centrality', 'Eigenvector_Centrality',
                                             'Avg_Shortest_Path_Length'])
reshaped_df.columns = [f"{metric}_Node{node}" for metric, node in reshaped_df.columns]
reshaped_df.reset_index(inplace=True)
reshaped_df.to_csv('Node_metrics.csv', index=False)